In [6]:
import numpy as np
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
# вариант построения эмбеддингов с word2vek
list_object = [
    'Смартфон Samsung Galaxy S21 128GB черный',
    'Galaxy S21 128GB Black (SM-G991BZKDSER)',
    'Наушники Apple AirPods Pro',
    'Беспроводные наушники AirPods Pro от Apple',
    'Телевизор LG OLED 55 дюймов',
    'iPhone 12 128GB Black',
    'Apple iPhone 12 128 ГБ black',
    'Ноутбук Lenovo IdeaPad 3 15.6'' 16GB RAM',
    'Lenovo IdeaPad 3/15.6"/Core i3 1215U/16/256/Win/Grey (82RK013WRK)'
]

# предобработка
sentences_token = [simple_preprocess(object) for object in list_object]

model = Word2Vec(
    sentences=sentences_token,
    vector_size=32,
    window=5,
    min_count=1,
    workers=1,
    seed=19
)

def get_sentence_vector(sentence_tokens):
    '''Получение векторного представления объекта
		Вход: предобработанные названия
    	Возвращает усреднённые эмбеддинги всех слов в названии'''
    vectors = [model.wv[word] for word in sentence_tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

sentence_vectors = np.array([get_sentence_vector(sent) for sent in sentences_token]) # применение функции

similarity_matrix = cosine_similarity(sentence_vectors) # матрица косинусного сходства

print('Матрица косинусного сходства:', np.round(similarity_matrix, 2), sep='\n')

threshold = 0.6 # порог для дубликатов

print('\nНайденные дубликаты:')

# проход по матрице для поиска дубликатов
for i in range(len(list_object)):
    for j in range(i+1, len(list_object)):
        if similarity_matrix[i][j] > threshold:
            print(f"'{list_object[i]}' ~ '{list_object[j]}' (score={similarity_matrix[i][j]:.2f})")

Матрица косинусного сходства:
[[ 1.    0.35  0.1   0.2  -0.04  0.35  0.17  0.02 -0.14]
 [ 0.35  1.   -0.03  0.03 -0.24  0.55  0.39  0.37  0.16]
 [ 0.1  -0.03  1.    0.82  0.06  0.11  0.23  0.06  0.05]
 [ 0.2   0.03  0.82  1.    0.12  0.04  0.04  0.22  0.05]
 [-0.04 -0.24  0.06  0.12  1.   -0.11 -0.1   0.11  0.18]
 [ 0.35  0.55  0.11  0.04 -0.11  1.    0.62  0.29  0.33]
 [ 0.17  0.39  0.23  0.04 -0.1   0.62  1.   -0.04  0.16]
 [ 0.02  0.37  0.06  0.22  0.11  0.29 -0.04  1.    0.43]
 [-0.14  0.16  0.05  0.05  0.18  0.33  0.16  0.43  1.  ]]

Найденные дубликаты:
'Наушники Apple AirPods Pro' ~ 'Беспроводные наушники AirPods Pro от Apple' (score=0.82)
'iPhone 12 128GB Black' ~ 'Apple iPhone 12 128 ГБ black' (score=0.62)


In [ ]:
# вариант построения эмбеддингов с BERT
model = SentenceTransformer('all-MiniLM-L6-v2')

similarity_matrix = cosine_similarity(model.encode(list_object))


print('Матрица косинусного сходства:', np.round(similarity_matrix, 2), sep='\n')

threshold = 0.6 # порог для дубликатов

print('\nНайденные дубликаты:')

for i in range(len(list_object)):
    for j in range(i+1, len(list_object)):
        if similarity_matrix[i][j] > threshold:
            print(f"'{list_object[i]}' ~ '{list_object[j]}' (score={similarity_matrix[i][j]:.2f})")

/home/alena/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Матрица косинусного сходства:
[[ 1.    0.68  0.31  0.32  0.57  0.38  0.38  0.38  0.24]
 [ 0.68  1.   -0.02 -0.03  0.25  0.56  0.53  0.14  0.3 ]
 [ 0.31 -0.02  1.    0.92  0.35  0.08  0.14  0.42  0.19]
 [ 0.32 -0.03  0.92  1.    0.36  0.05  0.11  0.44  0.18]
 [ 0.57  0.25  0.35  0.36  1.    0.14  0.2   0.36  0.23]
 [ 0.38  0.56  0.08  0.05  0.14  1.    0.87  0.15  0.28]
 [ 0.38  0.53  0.14  0.11  0.2   0.87  1.    0.13  0.23]
 [ 0.38  0.14  0.42  0.44  0.36  0.15  0.13  1.    0.64]
 [ 0.24  0.3   0.19  0.18  0.23  0.28  0.23  0.64  1.  ]]

Найденные дубликаты:
'Смартфон Samsung Galaxy S21 128GB черный' ~ 'Galaxy S21 128GB Black (SM-G991BZKDSER)' (score=0.68)
'Наушники Apple AirPods Pro' ~ 'Беспроводные наушники AirPods Pro от Apple' (score=0.92)
'iPhone 12 128GB Black' ~ 'Apple iPhone 12 128 ГБ black' (score=0.87)
'Ноутбук Lenovo IdeaPad 3 15.6 16GB RAM' ~ 'Lenovo IdeaPad 3/15.6"/Core i3 1215U/16/256/Win/Grey (82RK013WRK)' (score=0.64)
